# VAQ Evaluation Plots 2

LSQ++-style plotting notebook for the current VAQ grid.

This notebook follows the same flow as `lsqpp_relerr_plots.ipynb`:

- load aggregate `*_VAQ_adc_vs_exact_eval.csv` files
- aggregate by the full VAQ configuration
- produce one figure per dataset
- plot averaged trends separately from explicit configuration-legend trends
- save every figure as PDF and SVG

The output directory is `figures2` so these plots do not overwrite the earlier VAQ notebook outputs.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 40,
    "axes.labelsize": 20,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 21,
})
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

PROJECT_ROOT = Path("/home/cpanourg/projects/2-hdvc")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/vaq/vaq_bpv_grid_FIXED_20260503_150802")
FIGURES_DIR = DATA_DIR / "figures2"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

METHODS_TO_PLOT = ["VAQ"]
DATASETS_TO_PLOT = ["deep", "bigann", "gist", "msmarco", "openai"]
FIGURE_SAVE_FORMATS = ("pdf", "svg")

ADC_TIME_COL = "adc_time_per_pair_s"
ADC_TIME_LABEL = "ADC time (s)"
CONFIG_COLS = ["method", "dataset", "bits_per_vector", "min_bits", "max_bits", "variance"]


In [ ]:
# Color and marker palettes copied from the LSQ++ notebook.
COLOR_PALETTE = [
    "tab:blue", "tab:green", "tab:purple", "tab:orange", "tab:red",
    "tab:brown", "tab:pink", "tab:gray", "tab:olive", "tab:cyan",
]
MARKER_PALETTE = ["o", "v", "s", "^", "D", "<", ">", "p", "*", "h"]


def save_figure(fig, output_dir: Path, stem: str):
    output_dir.mkdir(parents=True, exist_ok=True)
    saved = []
    for ext in FIGURE_SAVE_FORMATS:
        path = output_dir / f"{stem}.{ext}"
        fig.savefig(path, bbox_inches="tight")
        saved.append(path)
    print("Saved " + " and ".join(str(p) for p in saved))


def style_axes(ax, tick_fontsize=40, grid_axis="y"):
    ax.grid(True, axis=grid_axis, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=tick_fontsize)


def adjust_ylabel_position(ax, y_label: str):
    if y_label == "Avg Relative Error":
        ax.yaxis.set_label_coords(-0.13, 0.39)


def set_sci_axes(ax):
    for axis in [ax.xaxis, ax.yaxis]:
        fmt = ScalarFormatter(useMathText=True)
        fmt.set_powerlimits((-2, 3))
        axis.set_major_formatter(fmt)


In [ ]:
def load_vaq_data(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    csv_paths = sorted(data_dir.glob("*_VAQ_adc_vs_exact_eval.csv"))
    if not csv_paths:
        raise FileNotFoundError(f"No VAQ CSV files found in {data_dir}")

    frames = []
    for path in csv_paths:
        df = pd.read_csv(path)
        if "dataset" not in df.columns:
            df["dataset"] = path.name.split("_")[0]
        if "method" not in df.columns:
            df["method"] = "VAQ"
        frames.append(df)

    df = pd.concat(frames, ignore_index=True)
    numeric_cols = [
        "bits_per_vector", "min_bits", "max_bits", "variance",
        "adc_time_s", "rel_error_mean", "rel_error_std",
        "train_time_s", "encoding_time_s", "distance_table_time_s", "cdist_time_s",
        "nb_sample", "nq_sample", "dim", "nb", "nq", "n_subquantizers", "nbits",
        "train_size", "seed",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["method"] = df["method"].replace({"VAQ": "VAQ"})
    df["pair_count"] = df["nb_sample"] * df["nq_sample"]
    df[ADC_TIME_COL] = df["adc_time_s"] / df["pair_count"]
    df["bpv_ratio"] = df["bits_per_vector"] / df["dim"]

    required = ["method", "dataset", "bits_per_vector", "min_bits", "max_bits", "variance", "adc_time_s", "rel_error_mean"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    return df


raw_df = load_vaq_data(DATA_DIR)
print("Rows per dataset:")
display(raw_df.groupby("dataset").size())
raw_df.sort_values(["dataset", "bits_per_vector", "min_bits", "max_bits"]).head(20)


In [ ]:
def aggregate_metric_df(df: pd.DataFrame, y_cols=("rel_error_mean", ADC_TIME_COL)) -> pd.DataFrame:
    agg_spec = {c: "mean" for c in y_cols if c in df.columns}
    for c in [
        "rel_error_std", "distance_table_time_s", "cdist_time_s",
        "train_time_s", "encoding_time_s", "dim", "nb", "nb_sample",
        "nq_sample", "pair_count", "adc_time_s", "bpv_ratio", "train_size",
    ]:
        if c in df.columns and c not in agg_spec:
            agg_spec[c] = "mean" if pd.api.types.is_numeric_dtype(df[c]) else "first"
    return (
        df.groupby(CONFIG_COLS, as_index=False)
        .agg(agg_spec)
        .sort_values(["dataset", "bits_per_vector", "min_bits", "max_bits"])
    )


plot_df = aggregate_metric_df(raw_df)
plot_df.groupby("dataset").size()


In [ ]:
def plot_avg_metric_vs_param(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    x_label: str,
    y_label: str,
    output_stem: str,
    datasets=DATASETS_TO_PLOT,
    methods=METHODS_TO_PLOT,
    output_dir: Path = FIGURES_DIR,
):
    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].dropna(subset=[x_col, y_col]).copy()
            if sub.empty:
                print(f"Skipping {method}/{dataset}: no rows")
                continue
            agg = sub.groupby([x_col], as_index=False)[y_col].agg(["mean", "std"]).reset_index().sort_values(x_col)
            fig, ax = plt.subplots()
            yerr = agg["std"].fillna(0)
            ax.errorbar(
                agg[x_col], agg["mean"], yerr=yerr,
                fmt="o-", color=COLOR_PALETTE[0], markersize=16,
                linewidth=2.5, markeredgewidth=2, markeredgecolor="black",
                capsize=5, capthick=2, elinewidth=1.5,
            )
            ax.set_xlabel(x_label, fontsize=40)
            ax.set_ylabel(y_label, fontsize=40)
            adjust_ylabel_position(ax, y_label)
            if x_col in ("bits_per_vector", "min_bits", "max_bits", "bpv_ratio", "variance"):
                vals = sorted(agg[x_col].dropna().unique())
                ax.set_xticks(vals)
                ax.set_xticklabels([
                    str(int(v)) if float(v).is_integer() else f"{v:.3g}" for v in vals
                ], rotation=0)
            style_axes(ax, tick_fontsize=34, grid_axis="y")
            set_sci_axes(ax)
            plt.tight_layout()
            stem = f"{output_stem}_{method.lower()}_{dataset}"
            save_figure(fig, output_dir, stem)
            plt.show()
            plt.close(fig)


In [ ]:
def _format_legend_value(col: str, value):
    if pd.isna(value):
        return f"{col}=NA"
    if col == "bits_per_vector":
        return f"bpv={int(value)}"
    if col == "min_bits":
        return f"min={int(value)}"
    if col == "max_bits":
        return f"max={int(value)}"
    if col == "variance":
        return f"var={value:.3g}"
    if isinstance(value, (int, np.integer)):
        return f"{col}={int(value)}"
    if isinstance(value, (float, np.floating)):
        return f"{col}={int(value)}" if float(value).is_integer() else f"{col}={value:.3g}"
    return f"{col}={value}"


def _legend_label(row, legend_cols):
    return ", ".join(_format_legend_value(col, row[col]) for col in legend_cols)


def plot_metric_with_config_legend(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    x_label: str,
    y_label: str,
    legend_cols: list,
    output_stem: str,
    datasets=DATASETS_TO_PLOT,
    methods=METHODS_TO_PLOT,
    output_dir: Path = FIGURES_DIR,
):
    for method in methods:
        for dataset in datasets:
            needed = [x_col, y_col] + legend_cols
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].dropna(subset=needed).copy()
            if sub.empty:
                print(f"Skipping {method}/{dataset}: no rows")
                continue
            agg = (
                sub.groupby([x_col] + legend_cols, as_index=False)[y_col]
                .mean()
                .sort_values(legend_cols + [x_col])
            )
            legend_settings = agg[legend_cols].drop_duplicates().sort_values(legend_cols).reset_index(drop=True)
            fig, ax = plt.subplots()
            for idx, setting in legend_settings.iterrows():
                mask = np.ones(len(agg), dtype=bool)
                for col in legend_cols:
                    mask &= agg[col].eq(setting[col])
                curve = agg.loc[mask].sort_values(x_col)
                color = COLOR_PALETTE[idx % len(COLOR_PALETTE)]
                marker = MARKER_PALETTE[idx % len(MARKER_PALETTE)]
                ax.plot(
                    curve[x_col], curve[y_col], marker=marker, linestyle="-",
                    color=color, markersize=12, linewidth=2.5,
                    markeredgewidth=1.5, markeredgecolor="black",
                    label=_legend_label(setting, legend_cols),
                )
            ax.set_xlabel(x_label, fontsize=40)
            ax.set_ylabel(y_label, fontsize=40)
            adjust_ylabel_position(ax, y_label)
            if x_col in ("bits_per_vector", "min_bits", "max_bits", "bpv_ratio", "variance"):
                vals = sorted(agg[x_col].dropna().unique())
                ax.set_xticks(vals)
                ax.set_xticklabels([
                    str(int(v)) if float(v).is_integer() else f"{v:.3g}" for v in vals
                ], rotation=0)
            style_axes(ax, tick_fontsize=34, grid_axis="y")
            set_sci_axes(ax)
            ax.legend(frameon=True, fontsize=13, loc="best")
            plt.tight_layout()
            stem = f"{output_stem}_{method.lower()}_{dataset}"
            save_figure(fig, output_dir, stem)
            plt.show()
            plt.close(fig)


## Avg Relative Error Plots


In [ ]:
# Averaged trends: same LSQ++ logic, one figure per dataset.
plot_avg_metric_vs_param(
    plot_df,
    x_col="bits_per_vector",
    y_col="rel_error_mean",
    x_label="Bits per vector",
    y_label="Avg Relative Error",
    output_stem="avg_relerr_vs_bits_per_vector",
)

plot_avg_metric_vs_param(
    plot_df,
    x_col="min_bits",
    y_col="rel_error_mean",
    x_label="Min bits",
    y_label="Avg Relative Error",
    output_stem="avg_relerr_vs_min_bits",
)

plot_avg_metric_vs_param(
    plot_df,
    x_col="max_bits",
    y_col="rel_error_mean",
    x_label="Max bits",
    y_label="Avg Relative Error",
    output_stem="avg_relerr_vs_max_bits",
)

plot_metric_with_config_legend(
    plot_df,
    x_col="bits_per_vector",
    y_col="rel_error_mean",
    x_label="Bits per vector",
    y_label="Avg Relative Error",
    legend_cols=["min_bits"],
    output_stem="avg_relerr_vs_bits_per_vector_legend_min_bits",
)

plot_metric_with_config_legend(
    plot_df,
    x_col="bits_per_vector",
    y_col="rel_error_mean",
    x_label="Bits per vector",
    y_label="Avg Relative Error",
    legend_cols=["max_bits"],
    output_stem="avg_relerr_vs_bits_per_vector_legend_max_bits",
)

plot_metric_with_config_legend(
    plot_df,
    x_col="bits_per_vector",
    y_col="rel_error_mean",
    x_label="Bits per vector",
    y_label="Avg Relative Error",
    legend_cols=["min_bits", "max_bits"],
    output_stem="avg_relerr_vs_bits_per_vector_with_config_legend",
)

plot_metric_with_config_legend(
    plot_df,
    x_col="min_bits",
    y_col="rel_error_mean",
    x_label="Min bits",
    y_label="Avg Relative Error",
    legend_cols=["max_bits"],
    output_stem="avg_relerr_vs_min_bits_legend_max_bits",
)

plot_metric_with_config_legend(
    plot_df,
    x_col="max_bits",
    y_col="rel_error_mean",
    x_label="Max bits",
    y_label="Avg Relative Error",
    legend_cols=["min_bits"],
    output_stem="avg_relerr_vs_max_bits_legend_min_bits",
)


## Min/Max Bit Pair Summary

These heatmaps average `rel_error_mean` across the available bit budgets for each `(min_bits, max_bits)` pair within each dataset. They are meant to diagnose which pair is stable, not to imply that every pair was run at every bit budget.


In [ ]:
def plot_minmax_pair_heatmaps(
    df: pd.DataFrame,
    metric_col: str = "rel_error_mean",
    output_stem: str = "avg_relerr_by_min_max_pair",
    datasets=DATASETS_TO_PLOT,
    methods=METHODS_TO_PLOT,
    output_dir: Path = FIGURES_DIR,
):
    summary_rows = []
    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].dropna(subset=["min_bits", "max_bits", metric_col]).copy()
            if sub.empty:
                print(f"Skipping {method}/{dataset}: no rows")
                continue
            pair_df = (
                sub.groupby(["min_bits", "max_bits"], as_index=False)
                .agg(mean_rel_error=(metric_col, "mean"), row_count=(metric_col, "size"))
                .sort_values(["mean_rel_error", "min_bits", "max_bits"])
            )
            best = pair_df.iloc[0]
            summary_rows.append({
                "dataset": dataset,
                "best_min_bits": int(best["min_bits"]),
                "best_max_bits": int(best["max_bits"]),
                "mean_rel_error": float(best["mean_rel_error"]),
                "rows_for_pair": int(best["row_count"]),
            })

            pivot = pair_df.pivot(index="min_bits", columns="max_bits", values="mean_rel_error").sort_index().sort_index(axis=1)
            fig, ax = plt.subplots(figsize=(7, 5.5))
            masked = np.ma.masked_invalid(pivot.values.astype(float))
            im = ax.imshow(masked, cmap="viridis_r", aspect="auto")
            ax.set_xticks(np.arange(len(pivot.columns)))
            ax.set_xticklabels([str(int(v)) for v in pivot.columns])
            ax.set_yticks(np.arange(len(pivot.index)))
            ax.set_yticklabels([str(int(v)) for v in pivot.index])
            ax.set_xlabel("Max bits", fontsize=34)
            ax.set_ylabel("Min bits", fontsize=34)
            ax.tick_params(axis="both", labelsize=26)
            for spine in ax.spines.values():
                spine.set_visible(False)
            for i, min_bit in enumerate(pivot.index):
                for j, max_bit in enumerate(pivot.columns):
                    val = pivot.loc[min_bit, max_bit]
                    if pd.notna(val):
                        ax.text(j, i, f"{val:.3g}", ha="center", va="center", color="white", fontsize=18, fontweight="bold")
            cbar = fig.colorbar(im, ax=ax)
            cbar.ax.tick_params(labelsize=22)
            cbar.set_label("Mean relative error", fontsize=24)
            plt.tight_layout()
            save_figure(fig, output_dir, f"{output_stem}_{method.lower()}_{dataset}")
            plt.show()
            plt.close(fig)
    return pd.DataFrame(summary_rows).sort_values("dataset")


if {"min_bits", "max_bits"}.issubset(plot_df.columns):
    best_minmax_pairs = plot_minmax_pair_heatmaps(plot_df)
    display(best_minmax_pairs)
else:
    print("Skipping min/max heatmap: columns min_bits/max_bits not available")


## Why Bits per Vector Can Look Worse

The fixed grid is not a clean one-dimensional sweep. At high `bits_per_vector`, many rows use `max_bits=16`, and those rows are often much worse than the `max_bits=8` rows. So an averaged `bits_per_vector` plot can increase because it is mixing different `(min_bits, max_bits)` pairs, not because more bits are intrinsically harmful. Use the config-legend plots and the min/max heatmaps above to compare like with like.


## ADC Time Plots


In [ ]:
plot_avg_metric_vs_param(
    plot_df,
    x_col="bits_per_vector",
    y_col=ADC_TIME_COL,
    x_label="Bits per vector",
    y_label=ADC_TIME_LABEL,
    output_stem="adc_time_vs_bits_per_vector",
)

plot_avg_metric_vs_param(
    plot_df,
    x_col="min_bits",
    y_col=ADC_TIME_COL,
    x_label="Min bits",
    y_label=ADC_TIME_LABEL,
    output_stem="adc_time_vs_min_bits",
)

plot_avg_metric_vs_param(
    plot_df,
    x_col="max_bits",
    y_col=ADC_TIME_COL,
    x_label="Max bits",
    y_label=ADC_TIME_LABEL,
    output_stem="adc_time_vs_max_bits",
)

plot_metric_with_config_legend(
    plot_df,
    x_col="bits_per_vector",
    y_col=ADC_TIME_COL,
    x_label="Bits per vector",
    y_label=ADC_TIME_LABEL,
    legend_cols=["min_bits", "max_bits"],
    output_stem="adc_time_vs_bits_per_vector_with_config_legend",
)

plot_metric_with_config_legend(
    plot_df,
    x_col="min_bits",
    y_col=ADC_TIME_COL,
    x_label="Min bits",
    y_label=ADC_TIME_LABEL,
    legend_cols=["max_bits"],
    output_stem="adc_time_vs_min_bits_legend_max_bits",
)

plot_metric_with_config_legend(
    plot_df,
    x_col="max_bits",
    y_col=ADC_TIME_COL,
    x_label="Max bits",
    y_label=ADC_TIME_LABEL,
    legend_cols=["min_bits"],
    output_stem="adc_time_vs_max_bits_legend_min_bits",
)


## Pareto Plots


In [ ]:
def pareto_frontier_minimize(df: pd.DataFrame, x_col: str, y_col: str) -> pd.DataFrame:
    pts = df.sort_values([x_col, y_col]).copy()
    frontier_rows = []
    best_y = np.inf
    for _, row in pts.iterrows():
        if row[y_col] < best_y:
            frontier_rows.append(row)
            best_y = row[y_col]
    if not frontier_rows:
        return pts.iloc[0:0]
    return pd.DataFrame(frontier_rows)


def plot_pareto(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    x_label: str,
    y_label: str,
    label_cols: list,
    output_stem: str,
    datasets=DATASETS_TO_PLOT,
    methods=METHODS_TO_PLOT,
    output_dir: Path = FIGURES_DIR,
):
    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].dropna(subset=[x_col, y_col]).copy()
            if sub.empty:
                print(f"Skipping {method}/{dataset}: no rows")
                continue
            frontier = pareto_frontier_minimize(sub, x_col, y_col)
            fig, ax = plt.subplots()
            ax.scatter(
                sub[x_col], sub[y_col],
                color="tab:blue", marker="o", s=260,
                edgecolors="black", linewidths=2, alpha=0.75,
            )
            if len(frontier) > 0:
                ax.plot(frontier[x_col], frontier[y_col], "-", color="gray", linewidth=1.5, alpha=0.8)
                ax.scatter(
                    frontier[x_col], frontier[y_col],
                    color="#D32F2F", marker="o", s=330,
                    edgecolors="black", linewidths=2, zorder=3,
                )
            for _, row in frontier.iterrows():
                label = ", ".join(_format_legend_value(col, row[col]) for col in label_cols if col in row.index)
                ax.annotate(
                    label, (row[x_col], row[y_col]),
                    xytext=(10, 10), textcoords="offset points", fontsize=13,
                    bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.72),
                    arrowprops=dict(arrowstyle="-", color="0.35", lw=0.8, shrinkA=0, shrinkB=5),
                )
            ax.set_xlabel(x_label, fontsize=40)
            ax.set_ylabel(y_label, fontsize=40)
            adjust_ylabel_position(ax, y_label)
            style_axes(ax, tick_fontsize=34, grid_axis="both")
            set_sci_axes(ax)
            plt.tight_layout()
            stem = f"{output_stem}_{method.lower()}_{dataset}"
            save_figure(fig, output_dir, stem)
            plt.show()
            plt.close(fig)


plot_pareto(
    plot_df,
    x_col=ADC_TIME_COL,
    y_col="rel_error_mean",
    x_label=ADC_TIME_LABEL,
    y_label="Avg Relative Error",
    label_cols=["bits_per_vector", "min_bits", "max_bits"],
    output_stem="pareto_relerr_vs_adc_time",
)
